<a href="https://colab.research.google.com/github/CodeMaverick-143/ML_Work/blob/main/LangChain_basic_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y langchain langchain-core langchain-groq
!pip install --upgrade langchain==0.2.14 langchain-core==0.2.43 langchain-groq==0.1.5

Found existing installation: langchain 0.2.14
Uninstalling langchain-0.2.14:
  Successfully uninstalled langchain-0.2.14
Found existing installation: langchain-core 0.2.43
Uninstalling langchain-core-0.2.43:
  Successfully uninstalled langchain-core-0.2.43
Found existing installation: langchain-groq 0.1.5
Uninstalling langchain-groq-0.1.5:
  Successfully uninstalled langchain-groq-0.1.5
  Using cached langchain-0.2.14-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_groq-0.1.5-py3-none-any.whl.metadata (2.8 kB)
Using cached langchain-0.2.14-py3-none-any.whl (997 kB)
Using cached langchain_core-0.2.43-py3-none-any.whl (397 kB)
Using cached langchain_groq-0.1.5-py3-none-any.whl (11 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.8 requires langchain-core>

In [2]:
# !pip uninstall -y langgraph-prebuilt langgraph

In [3]:
# !pip install -q --upgrade langchain langchain-groq

In [4]:
import os
from google.colab import userdata
from langchain.prompts import PromptTemplate
from langchain_groq import ChatGroq

# Retrieve the secret value as a string
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API')


In [5]:
# Test if your api key works

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")

print(llm.invoke("Explain linear regression in one paragraph"))

content='Linear regression is a statistical model that predicts the value of a continuous outcome variable based on one or more predictor variables. The goal of linear regression is to create a linear equation that best predicts the relationship between the predictor variables and the outcome variable. The equation takes the form of Y = β0 + β1X + ε, where Y is the outcome variable, X is the predictor variable, β0 is the intercept or constant term, β1 is the slope coefficient, and ε is the error term. The model is "fitted" to the data by estimating the values of β0 and β1 that minimize the difference between the observed and predicted values of Y. The resulting equation can be used to make predictions for new, unseen data, and the coefficients can be interpreted to understand the relationship between the predictor variables and the outcome variable, including the direction and strength of the relationship.' response_metadata={'token_usage': {'completion_tokens': 178, 'prompt_tokens': 4

# LangChain Exercise: Manual LLM Chaining with ChatGroq

## Objective

In this exercise, you will build a **multi-step LLM pipeline** using LangChain — without relying on high-level chain abstractions.

You will:
1. Generate an explanation of a topic  
2. Use that explanation to create quiz questions  

This demonstrates how **outputs from one LLM call can be used as inputs to another**.

Instead of using `LLMChain` or `SequentialChain`, you will implement this flow **manually** using `.invoke()`.

---

## Key Components Used

### 1. `ChatGroq`
- Interface to large language models via Groq
- Extremely fast inference
- Used to generate responses from prompts

### 2. `PromptTemplate`
- Used to create **structured and reusable prompts**
- Allows you to insert variables dynamically into prompts
- Helps maintain clean and consistent prompt design


### 3. `format()`
- Fills in the variables inside a PromptTemplate
- Converts the template into a final prompt string that can be sent to the LLM

### 4. `invoke()`
- Sends the formatted prompt to the LLM
- Executes the model and returns a response object

### 5. `content()`
- Extracts the actual text output from the response object
- This is what you use for the next step in the pipeline


`Input` → `Prompt` → `LLM` → `Output` → `Next Prompt` → `LLM` → `Final Output`

In [6]:
from langchain.prompts import PromptTemplate
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

prompt1 = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms"
)

prompt2 = PromptTemplate(
    input_variables=["explanation"],
    template="Create 2 quiz questions from this:\n{explanation}"
)

formatted_prompt1 = prompt1.format(topic="KNN")
response1 = llm.invoke(formatted_prompt1)
explanation = response1.content if hasattr(response1, "content") else str(response1)

formatted_prompt2 = prompt2.format(explanation=explanation)
response2 = llm.invoke(formatted_prompt2)
quiz = response2.content if hasattr(response2, "content") else str(response2)

result = {
    "explanation": explanation,
    "quiz": quiz
}

print(result)

{'explanation': '**K-Nearest Neighbors (KNN) Algorithm**\n\nKNN is a simple and intuitive machine learning algorithm used for classification and regression tasks. Here\'s how it works:\n\n**Key Idea:**\nThe KNN algorithm assumes that similar data points are likely to have similar outcomes. It looks at the "nearest neighbors" of a new, unseen data point to predict its outcome.\n\n**How it Works:**\n\n1. **Data Collection**: Gather a dataset with input features and corresponding outcomes (labels).\n2. **New Data Point**: Receive a new, unseen data point with input features, but no outcome (label).\n3. **Distance Calculation**: Calculate the distance between the new data point and all existing data points in the dataset.\n4. **K-Nearest Neighbors**: Select the top K data points with the smallest distances to the new data point. These are the "nearest neighbors."\n5. **Outcome Prediction**: Look at the outcomes (labels) of the K nearest neighbors and use them to predict the outcome for the